---
title: "Sektorale Energieallokation – Transition 2040"
lang: de
format:
  html:
    code-fold: true
    code-summary: "Berechnung anzeigen"
execute:
  echo: true
  warning: false
jupyter: python3
---

# Quantitative sektorale Allokation {#sec-allokation-notebook}

Stand: 29. Oktober 2026. Dieser Anhang berechnet die Top-down-Grundlage der PEQ-Kalibrierung: den nationalen elektrischen Gebäudesektorsaldo einschließlich MIV und ein einmalig zugeteilte zentrale Restdeckung. Alle fachlichen Rechnungen stehen in diesen Zellen. Die Gebäudetypologien werden im Gebäudesektormodell abgeglichen.

**Quellen:** UBA REP-0880, Tabelle 15, S. 60 (Strombedarf) und Tabelle 16, S. 63 (Erzeugung/Nettohandel), Ausgabe ©2024, Titelblatt 2023 [@uba2023transition]. Die Zuordnungslogik folgt der in @sec-allokation beschriebenen Weiterentwicklung der historischen Methode [@schneider2020lokal; @schneider2023context]. Die nachfolgenden Parameter sind ausdrücklich eigene Rechenannahmen, keine vom UBA veröffentlichten Verkehrs- oder PV-Anteile.

## 1. Bilanzgrenze und Einheiten

Alle Flüsse sind jährliche **elektrische Endenergie in PJ/a**; 1 PJ = 1/3,6 TWh. Positive Salden bedeuten Deckung minus Bedarf. Gebäude umfassen hier die UBA-Betriebsposition sowie den zugeordneten MIV. Nicht-elektrische Wärme, Brennstoffe, graue Energie und Primärenergie werden nicht mit Strom addiert. Ein virtuelles MIV-Budget ist eine Zuordnung vorhandener Erzeugung, keine zusätzliche Energiequelle.


In [1]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'_quarto.yml').exists())
DATA = ROOT/'Daten/sektorenmodell'
OUT = ROOT/'Berechnungen/sektorenmodell/outputs/allokation'
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi':130, 'font.size':10, 'axes.spines.top':False, 'axes.spines.right':False})
tables={name:pd.read_csv(DATA/name) for name in ['stromerzeugung_2040.csv','strombedarf_2040.csv']}
selected={name:df.loc[df.scenario.eq('transition_2040')].copy() for name,df in tables.items()}
for name,df in selected.items():
    assert df.metric.is_unique and df.value_pj.notna().all()
    assert df.year.eq(2040).all() and df.source_id.eq('TRANSITION').all()
    display(df[['metric','value_pj','energy_basis','source_location']])
g=selected['stromerzeugung_2040.csv'].set_index('metric').value_pj.to_dict()
d=selected['strombedarf_2040.csv'].set_index('metric').value_pj.to_dict()
provenance={name:hashlib.sha256((DATA/name).read_bytes()).hexdigest() for name in tables}


,metric,value_pj,energy_basis,source_location
18,fossil,1,electricity_generation_or_net_import,"Tabelle 16, S. 63"
19,hydro,173,electricity_generation_or_net_import,"Tabelle 16, S. 63"
20,biomass,21,electricity_generation_or_net_import,"Tabelle 16, S. 63"
21,geothermal,4,electricity_generation_or_net_import,"Tabelle 16, S. 63"
22,pv,149,electricity_generation_or_net_import,"Tabelle 16, S. 63"
23,wind,105,electricity_generation_or_net_import,"Tabelle 16, S. 63"
24,hydrogen_electricity,4,electricity_generation_or_net_import,"Tabelle 16, S. 63"
25,generation_published_total,457,electricity_generation_or_net_import,"Tabelle 16, S. 63"
26,net_imports,-7,electricity_generation_or_net_import,"Tabelle 16, S. 63"


,metric,value_pj,energy_basis,source_location
16,verkehr,90,electricity_demand,"Tabelle 15, S. 60"
17,gebaeude,108,electricity_demand,"Tabelle 15, S. 60"
18,landwirtschaft,4,electricity_demand,"Tabelle 15, S. 60"
19,industrie,140,electricity_demand,"Tabelle 15, S. 60"
20,verbrauch_sektor_energie,34,electricity_demand,"Tabelle 15, S. 60"
21,transportverluste,19,electricity_demand,"Tabelle 15, S. 60"
22,umwandlungseinsatz,57,electricity_demand,"Tabelle 15, S. 60"
23,electricity_demand_published_total,450,electricity_demand,"Tabelle 15, S. 60"


## 2. Quellbilanz vor Allokation

$R_{Quelle}=G+I-D$. Nettoimporte $I<0$ sind Exporte. Die gedruckte Gesamtsumme des Bedarfs wird neben der Summe der Komponenten erhalten. Die Zuordnung arbeitet mit den Komponenten, ohne ihren Unterschied zu verteilen.


In [2]:
generation_keys=['hydro','wind','biomass','geothermal','hydrogen_electricity','pv','fossil']
demand_keys=['verkehr','gebaeude','industrie','landwirtschaft','verbrauch_sektor_energie','transportverluste','umwandlungseinsatz']
source_generation=sum(g[k] for k in generation_keys)
source_demand=sum(d[k] for k in demand_keys)
source_residual=source_generation+g['net_imports']-source_demand
assert source_generation==457 and source_demand==452 and source_residual==-2
display(pd.DataFrame({'Position':['Erzeugung','Nettoimporte','Bedarf publiziert','Bedarf Komponenten','Rest Komponenten'],
                      'PJ_el/a':[source_generation,g['net_imports'],d['electricity_demand_published_total'],source_demand,source_residual]}))


,Position,PJ_el/a
0,Erzeugung,457
1,Nettoimporte,-7
2,Bedarf publiziert,450
3,Bedarf Komponenten,452
4,Rest Komponenten,-2


## 3. Modellannahmen vom 29.10.2026

Freiflächen-PV gehört zu den zentralen Großkraftwerken. Gebäude-PV umfasst Eigenverbrauch und Einspeisung. H₂-Rückverstromung (4 PJ/a) und fossiler Strom (1 PJ/a) werden vollständig als Deckung berücksichtigt; damit wird fossiler Strom nicht als erneuerbar bezeichnet.

Der Nettoexport wird für dieses Modell mit **5 statt 7 PJ/a** angesetzt: 457 PJ Erzeugung = 452 PJ Bedarfskomponenten + 5 PJ Export. Die Quelldaten bleiben unverändert. Der Quellrest beträgt −2 PJ/a, der Modellrest null. Dies ist eine Modellannahme zur Bilanzschließung, keine Korrektur der UBA-Quelle.

Die Referenz setzt 0 % Freiflächen-PV und 80/20/0 % MIV/ÖPNV/übrigen Verkehr. Verkehrsanteile beziehen sich auf den elektrischen Energiebedarf des Verkehrs insgesamt. Industrie, Landwirtschaft, Energiebranche, Umwandlung, Verluste und Export sowie Nicht-MIV-Verkehr werden einmalig abgezogen. Der Elektrolysebedarf bleibt enthalten.


In [3]:
assumptions = dict(date='2026-10-29', scenario='transition_2040',
    source_net_exports_pj=7, model_net_exports_pj=5, export_adjustment_pj=-2,
    reason='Dokumentierte Modellannahme zur Bilanzschließung bei 452 PJ Bedarfskomponenten',
    hydrogen_electricity_included_pj=4, fossil_included_pj=1)
parameters=dict(other_pv_share=0.,miv_share=.8,public_transport_share=.2,other_transport_share=0.,
    source='Modellannahmen 29.10.2026; PV- und Verkehrsanteile sind Varianten',
    pe_factor_electricity=None,pe_factor_year=None,pe_factor_source=None)
model_residual=source_generation-assumptions['model_net_exports_pj']-source_demand
assert np.isclose(model_residual,0)
display(pd.DataFrame([assumptions]).T)


,0
date,2026-10-29
scenario,transition_2040
source_net_exports_pj,7
model_net_exports_pj,5
export_adjustment_pj,-2
reason,Dokumentierte Modellannahme zur Bilanzschließu...
hydrogen_electricity_included_pj,4
fossil_included_pj,1


## 4. Zentrale Großkraftwerke und Restdeckung

$$C=\sum_{i\ne PV}G_i+fG_{PV}+I_+,\qquad R=C-F-(o+r)D_{Verkehr}.$$

$F$ enthält die priorisierten Bedarfe einschließlich Modell-Nettoexport. Die Restdeckung $R$ wird vollständig Gebäude + MIV zugeordnet, ohne Obergrenze durch den MIV-Bedarf. Ein negatives $R$ wird als zentraler Fehlbetrag separat dargestellt und durch den Gebäude-PV-Überschuss ausgeglichen.

## 5. Vollständige Zuordnungsprüfung

$$B_{Geb+MIV}=(1-f)G_{PV}+R-D_{Geb}-mD_{Verkehr}=R_{Modell}=0.$$

Für die Anzeige gilt: verfügbare Restdeckung $=\max(R,0)$, zentraler Fehlbetrag $=\max(-R,0)$. Der Fehlbetrag darf nicht durch Kappung verschwinden. Gebäude-PV plus verfügbare Restdeckung decken Gebäude, MIV und gegebenenfalls diesen zentralen Fehlbetrag.


In [4]:
def allocate(p):
    assert 'eligible_nonpv_share' not in p, 'Veraltete Anrechnungsparameter: volle Deckungsanrechnung ist festgelegt'
    for key in ['other_pv_share','miv_share','public_transport_share','other_transport_share']:
        assert np.isfinite(p[key]) and 0<=p[key]<=1, key
    assert np.isclose(sum(p[k] for k in ['miv_share','public_transport_share','other_transport_share']),1)
    pv_other=g['pv']*p['other_pv_share']; pv_building=g['pv']-pv_other
    central=sum(g[k] for k in generation_keys if k!='pv')+pv_other+max(g['net_imports'],0)
    fixed=sum(d[k] for k in ['industrie','landwirtschaft','verbrauch_sektor_energie','umwandlungseinsatz','transportverluste'])+assumptions['model_net_exports_pj']
    miv=d['verkehr']*p['miv_share']; public=d['verkehr']*p['public_transport_share']; other=d['verkehr']*p['other_transport_share']
    credit=central-fixed-public-other
    operation=pv_building-d['gebaeude']; balance=operation+credit-miv
    assert np.isclose(balance,model_residual), 'Zuordnungsbilanz verletzt'
    return dict(central_pj=central,fixed_priority_pj=fixed,public_transport_pj=public,other_transport_pj=other,
        miv_demand_pj=miv,central_credit_pj=credit,available_credit_pj=max(credit,0),central_shortfall_pj=max(-credit,0),
        building_pv_pj=pv_building,other_pv_pj=pv_other,building_operation_pj=operation,
        building_target_pj=balance,source_residual_pj=source_residual,model_residual_pj=model_residual,
        allocation_residual_pj=balance)
result=allocate(parameters)
result_table=pd.DataFrame([{'Position':k,'PJ_el/a':v,'TWh_el/a':v/3.6} for k,v in result.items()])
display(result_table.round(4))


,Position,PJ_el/a,TWh_el/a
0,central_pj,308.0,85.5556
1,fixed_priority_pj,259.0,71.9444
2,public_transport_pj,18.0,5.0000
3,other_transport_pj,0.0,0.0000
4,miv_demand_pj,72.0,20.0000
5,central_credit_pj,31.0,8.6111
6,available_credit_pj,31.0,8.6111
7,central_shortfall_pj,0.0,0.0000
8,building_pv_pj,149.0,41.3889
9,other_pv_pj,0.0,0.0000


## 6. Varianten

PV gesamt = Gebäude-PV + Freiflächen-PV; Verkehr gesamt = MIV + ÖPNV + übriger Verkehr. Freiflächen-PV verschiebt Deckung zwischen Gebäuden und zentralen Großkraftwerken. Der gemeinsame Top-Down-Saldo bleibt null. Negative zentrale Reste sind ein Rückgriff auf den Gebäude-PV-Überschuss.


In [5]:
import copy, sys
sys.path.insert(0,str(ROOT))
from Berechnungen.methodik_plots import allocation_figures
from IPython.display import HTML
variants=[]
for label,f,m in [('Referenz',0,.8),('20 % Freiflächen-PV',.2,.8),('60 % MIV',0,.6),('40 % MIV',0,.4),('100 % Freiflächen-PV',1,.8)]:
    p=dict(parameters,other_pv_share=f,miv_share=m,public_transport_share=1-m,other_transport_share=0)
    r=allocate(p)
    variants.append({'Variante':label,'PV gesamt':g['pv'],'PV Gebäude':r['building_pv_pj'],'PV Freifläche':r['other_pv_pj'],
        'Verkehr gesamt':d['verkehr'],'ÖPNV':r['public_transport_pj'],'MIV':r['miv_demand_pj'],
        'Übriger Verkehr':r['other_transport_pj'],'MIV-Anteil [%]':100*m,'Freiflächen-Anteil [%]':100*f,
        'Gebäude':d['gebaeude'],'Restdeckung':r['central_credit_pj'],'Verfügbare Restdeckung':r['available_credit_pj'],
        'Saldo':r['building_target_pj'],'Zentraler Fehlbetrag':r['central_shortfall_pj']})
comparison=pd.DataFrame(variants)
pv_columns=['Variante','PV gesamt','PV Gebäude','PV Freifläche','Freiflächen-Anteil [%]']
traffic_columns=['Variante','Verkehr gesamt','ÖPNV','MIV','Übriger Verkehr','MIV-Anteil [%]']
result_columns=['Variante','Restdeckung','Saldo','Zentraler Fehlbetrag']
for cols in [pv_columns,traffic_columns,result_columns]: display(comparison[cols].round(3))
steps={'Zentrale Großkraftwerke':result['central_pj'],'Industrie':-d['industrie'],
    'Landwirtschaft':-d['landwirtschaft'],'Energiebranche':-d['verbrauch_sektor_energie'],
    'Umwandlung':-d['umwandlungseinsatz'],'Verluste':-d['transportverluste'],
    'Nettoexport (Modell)':-assumptions['model_net_exports_pj'],'ÖPNV':-result['public_transport_pj'],
    'Übriger Verkehr':-result['other_transport_pj'],'Restdeckung':result['central_credit_pj']}
allocation_figures(variants,steps)
for name in ['zentraler_pool','allokation_deckung','allokation_saldo']:
    display(HTML(f'<iframe title="{name}" src="../../Abbildungen/methodik/{name}.html" style="width:100%;height:700px;border:0"></iframe>'))


,Variante,PV gesamt,PV Gebäude,PV Freifläche,Freiflächen-Anteil [%]
0,Referenz,149,149.0,0.0,0.0
1,20 % Freiflächen-PV,149,119.2,29.8,20.0
2,60 % MIV,149,149.0,0.0,0.0
3,40 % MIV,149,149.0,0.0,0.0
4,100 % Freiflächen-PV,149,0.0,149.0,100.0


,Variante,Verkehr gesamt,ÖPNV,MIV,Übriger Verkehr,MIV-Anteil [%]
0,Referenz,90,18.0,72.0,0,80.0
1,20 % Freiflächen-PV,90,18.0,72.0,0,80.0
2,60 % MIV,90,36.0,54.0,0,60.0
3,40 % MIV,90,54.0,36.0,0,40.0
4,100 % Freiflächen-PV,90,18.0,72.0,0,80.0


,Variante,Restdeckung,Saldo,Zentraler Fehlbetrag
0,Referenz,31.0,0.0,0.0
1,20 % Freiflächen-PV,60.8,0.0,0.0
2,60 % MIV,13.0,0.0,0.0
3,40 % MIV,-5.0,0.0,5.0
4,100 % Freiflächen-PV,180.0,0.0,0.0


C:\Users\schneids\AppData\Roaming\Python\Python313\site-packages\IPython\core\display.py:447: UserWarning:

Consider using IPython.display.IFrame instead



## 7. Primärenergie und nutzungsbezogene Gutschrift

Die sektorale Zuordnung bleibt in elektrischer Endenergie 2040. Für die spätere Bewertung mit heutigen Konversionsfaktoren gilt $E_{PE,MIV}=R\,f_{PE,Strom}$. Das ist die Bewertung des zugeordneten Strombudgets, keine Prognose des PE-Faktors 2040. Eine vollständige physische PE-Quartiersbilanz benötigt zusätzlich die übrigen Energieträger und die zeitlichen Faktoren. Ein jährlicher Ersatzfaktor ist nur bei dokumentierter Kompatibilität mit dem Bewertungsverfahren zulässig; monatliche Gewichtungen können einen einzelnen Jahresfaktor unzureichend machen.

Hier liegt kein freigegebener heutiger Faktorensatz vor. PE-Budget und spezifische Gutschrift bleiben deshalb **offen**, nicht null. Auch Nutzungsanteile und Ziel-NGF sind nicht neu gesetzt. Die historische Gutschrift 2023 wird durch das elektrische Rechenbeispiel nicht ersetzt.


In [6]:
factor=parameters['pe_factor_electricity']
if factor is None:
    primary_energy_budget=None
else:
    assert np.isfinite(factor) and factor>=0
    assert parameters['pe_factor_year'] and parameters['pe_factor_source']
    primary_energy_budget=result['central_credit_pj']*factor
display(pd.DataFrame([{'Zentrale Restdeckung [PJ/a]':result['central_credit_pj'],
                      'PE-Budget [PJ_PE/a]':primary_energy_budget,
                      'Status':'Offen: belegter Faktorensatz und nutzungsbezogene Verteilung fehlen' if factor is None else 'Parametrierte Umrechnung'}]))


,Zentrale Restdeckung [PJ/a],PE-Budget [PJ_PE/a],Status
0,31.0,None,Offen: belegter Faktorensatz und nutzungsbezog...


## 8. Kontrollen und Ausgaben

Die Kontrollen prüfen PV-Erhaltung, Verkehrsanteile, vollständige Zuordnung und den Modell-Bilanzschluss einschließlich negativer zentraler Reste. JSON-Schema 2 ersetzt das gekappte `miv_budget_pj` durch `central_credit_pj`; alle abhängigen Rechnungen verwenden die neue Größe. Quellen-Hashes, Quellrest und Exportannahme bleiben nachvollziehbar. PE-Faktorensatz und nutzungsbezogene Verteilung sind weiterhin offen.


In [7]:
for f in [0,.2,1]:
    for m in [0,.4,.6,.8,1]:
        r=allocate(dict(parameters,other_pv_share=f,miv_share=m,public_transport_share=1-m))
        assert np.isclose(r['building_pv_pj']+r['other_pv_pj'],g['pv'])
        assert np.isclose(r['allocation_residual_pj'],0)
        assert np.isclose(r['central_credit_pj'],90*m-41+149*f)
assert np.isclose(result['central_credit_pj'],31)
for bad in [-.01,1.01]:
    p=copy.deepcopy(parameters);p['other_pv_share']=bad
    try: allocate(p)
    except AssertionError: pass
    else: raise AssertionError('Ungültiger PV-Anteil akzeptiert')
payload={'schema_version':2,'assumptions':assumptions,'parameters':parameters,'result':result,'pe_budget_pj':primary_energy_budget,'source_sha256':provenance}
(OUT/'allokation.json').write_text(json.dumps(payload,ensure_ascii=False,indent=2),encoding='utf-8')
result_table.to_csv(OUT/'referenzfall.csv',index=False)
comparison.to_csv(OUT/'varianten.csv',index=False)
def table(frame):
    frame=frame.round(3).copy()
    if 'Vorrang gedeckt' in frame: frame['Vorrang gedeckt']=frame['Vorrang gedeckt'].map({True:'ja',False:'nein'})
    return '| '+' | '.join(frame.columns)+' |\n| '+' | '.join(['---']*len(frame.columns))+' |\n'+'\n'.join('| '+' | '.join(map(str,row))+' |' for row in frame.itertuples(index=False,name=None))
text=table(comparison[pv_columns])+'\n\n: PV-Aufteilung in PJ Strom/a; Freiflächenanteil bezogen auf die gesamte PV-Erzeugung. {#tbl-allokation-pv}\n\n'
text+=table(comparison[traffic_columns])+'\n\n: Verkehrsaufteilung in PJ Strom/a; MIV-Anteil bezogen auf den elektrischen Verkehrsbedarf. {#tbl-allokation-verkehr}\n\n'
text+=table(comparison[result_columns])+'\n\n: Allokationsergebnisse in PJ Strom/a. Restdeckung ist vorzeichenbehaftet; ein zentraler Fehlbetrag wird durch Gebäude-PV ausgeglichen. {#tbl-allokation-vorlaeufig}\n'
(ROOT/'Kapitel/_generated/allokation-ergebnisse.qmd').write_text(text,encoding='utf-8')
print('Bilanz- und Grenzfallkontrollen bestanden.')


Bilanz- und Grenzfallkontrollen bestanden.
